# Session 2: Control Flow & Functions

**Course:** Python for Data Engineering  
**Phase 1:** Foundations

**What we'll cover:**
- if-else conditions
- Loops (for, while)
- Functions, arguments, return values
- Lambda basics
- Lab: Filtering, aggregation, sales data summarizer

---

## 1. if-else Conditions

Used everywhere in pipelines — deciding what to do with a record based on its values.

```python
if condition:
    # do this
elif another_condition:
    # do that
else:
    # default
```

In [ ]:
# Basic if-else

row_count = 15000

if row_count == 0:
    print("No data received!")
elif row_count < 1000:
    print("Small batch")
elif row_count < 10000:
    print("Medium batch")
else:
    print("Large batch")

In [ ]:
# Real use case: validating a record before loading

record = {"name": "Alice", "email": "alice@example.com", "age": 28, "salary": -5000}

# Check multiple conditions
if not record["name"]:
    print("SKIP: name is empty")
elif "@" not in record["email"]:
    print("SKIP: invalid email")
elif record["age"] < 0 or record["age"] > 120:
    print("SKIP: invalid age")
elif record["salary"] < 0:
    print("SKIP: negative salary")
else:
    print("VALID: ready to load")

In [ ]:
# Nested conditions + combining with and/or

status = "completed"
row_count = 5000
has_errors = False

if status == "completed" and row_count > 0 and not has_errors:
    print("Pipeline run successful")
elif status == "completed" and has_errors:
    print("Completed with errors — check logs")
else:
    print("Pipeline failed")

**Try it:** Write a data quality checker.

Given a record, check:
- `amount` should be > 0
- `status` should be one of: `"active"`, `"pending"`, `"completed"`
- `email` should contain `"@"`

Print whether the record is valid or which check failed.

In [ ]:
record = {"amount": 150.0, "status": "active", "email": "bob@company.com"}

# Your code here


---

## 2. Loops

### 2.1 for loops

Iterate over lists, dicts, ranges — the bread and butter of batch processing.

In [ ]:
# Looping through a list

files = ["sales.csv", "users.csv", "products.csv"]

for file in files:
    print(f"Processing {file}...")

In [ ]:
# enumerate - when you need the index too

for i, file in enumerate(files):
    print(f"  [{i+1}/{len(files)}] Processing {file}")

In [ ]:
# Looping through a dictionary

config = {"host": "localhost", "port": 5432, "db": "sales"}

for key, value in config.items():
    print(f"  {key} = {value}")

In [ ]:
# range - useful for generating batch numbers, IDs, etc.

# Process records in batches
total = 5000
batch_size = 1000

for start in range(0, total, batch_size):
    end = min(start + batch_size, total)
    print(f"  Processing rows {start} to {end}")

In [ ]:
# Processing a batch of records (very common pattern)

records = [
    {"name": "Alice", "amount": 120.50, "status": "completed"},
    {"name": "Bob", "amount": -10.00, "status": "failed"},
    {"name": "Charlie", "amount": 89.99, "status": "completed"},
    {"name": "Diana", "amount": 0, "status": "completed"},
]

valid = []
invalid = []

for record in records:
    if record["status"] == "completed" and record["amount"] > 0:
        valid.append(record)
    else:
        invalid.append(record)

print(f"Valid: {len(valid)}, Invalid: {len(invalid)}")
for r in valid:
    print(f"  {r['name']}: ${r['amount']}")

### break, continue, else

- `break` — stop the loop entirely
- `continue` — skip to the next iteration
- `else` on a loop — runs only if the loop finished without a `break`

In [ ]:
# continue - skip bad records

data = [10, 20, None, 30, None, 40]
total = 0

for value in data:
    if value is None:
        continue   # skip nulls
    total += value

print(f"Total (skipping nulls): {total}")

In [ ]:
# break - stop when you find a critical error

logs = ["INFO: started", "INFO: loaded", "ERROR: connection failed", "INFO: done"]

for log in logs:
    print(f"Reading: {log}")
    if log.startswith("ERROR"):
        print("Critical error found — stopping.")
        break

### List comprehensions

A shortcut for building lists with a loop. Very Pythonic.

In [ ]:
# Regular loop
amounts = [120.50, 89.99, 250.00, 45.30]

with_tax = []
for a in amounts:
    with_tax.append(round(a * 1.08, 2))
print(with_tax)

# Same thing as a list comprehension — one line
with_tax = [round(a * 1.08, 2) for a in amounts]
print(with_tax)

# With a filter
big_orders = [a for a in amounts if a > 100]
print(f"Orders over $100: {big_orders}")

# Extract names from records
records = [{"name": "Alice", "age": 30}, {"name": "Bob", "age": 25}]
names = [r["name"] for r in records]
print(names)

### 2.2 while loops

Less common than `for`, but useful for retries, polling, and waiting for conditions.

In [ ]:
# Retry pattern — very common in data pipelines

import random

max_retries = 5
attempt = 0
success = False

while attempt < max_retries and not success:
    attempt += 1
    print(f"Attempt {attempt}...", end=" ")
    
    # Simulate: 40% chance of success
    if random.random() < 0.4:
        success = True
        print("Success!")
    else:
        print("Failed, retrying...")

if not success:
    print(f"Gave up after {max_retries} attempts")

**Try it:**

Given this list of orders:
```python
orders = [
    {"id": 1, "amount": 250, "region": "north"},
    {"id": 2, "amount": 80, "region": "south"},
    {"id": 3, "amount": 120, "region": "north"},
    {"id": 4, "amount": 45, "region": "east"},
    {"id": 5, "amount": 300, "region": "south"},
    {"id": 6, "amount": 60, "region": "north"},
]
```

1. Loop through and print only orders where `amount > 100`
2. Use a list comprehension to get all amounts from "north" region
3. Calculate total amount for each region using a loop

In [ ]:
orders = [
    {"id": 1, "amount": 250, "region": "north"},
    {"id": 2, "amount": 80, "region": "south"},
    {"id": 3, "amount": 120, "region": "north"},
    {"id": 4, "amount": 45, "region": "east"},
    {"id": 5, "amount": 300, "region": "south"},
    {"id": 6, "amount": 60, "region": "north"},
]

# 1. Print orders with amount > 100

# 2. List comprehension: amounts from north region

# 3. Total per region (hint: use a dictionary to accumulate)


---

## 3. Functions

Functions let you write a piece of logic once and reuse it. In data engineering, you'll write functions for cleaning, validating, transforming — anything you do repeatedly.

```python
def function_name(parameters):
    # do something
    return result
```

In [ ]:
# Basic function

def clean_name(raw_name):
    return raw_name.strip().title()

print(clean_name("  alice JOHNSON  "))
print(clean_name("BOB smith"))

In [ ]:
# Function with multiple parameters and a default value

def clean_currency(value, currency_symbol="$"):
    """Remove currency symbol and commas, return float."""
    cleaned = value.replace(currency_symbol, "").replace(",", "")
    return float(cleaned)

print(clean_currency("$1,299.99"))       # uses default $
print(clean_currency("€2.500,00", "€"))  # custom symbol

In [ ]:
# Returning multiple values

def validate_record(record):
    """Check if a record is valid. Returns (is_valid, reason)."""
    if not record.get("name"):
        return False, "missing name"
    if record.get("amount", 0) < 0:
        return False, "negative amount"
    if record.get("status") not in ["active", "completed"]:
        return False, f"invalid status: {record.get('status')}"
    return True, "ok"

# Test it
test_records = [
    {"name": "Alice", "amount": 100, "status": "active"},
    {"name": "", "amount": 50, "status": "active"},
    {"name": "Bob", "amount": -10, "status": "active"},
    {"name": "Charlie", "amount": 200, "status": "unknown"},
]

for rec in test_records:
    valid, reason = validate_record(rec)
    status = "PASS" if valid else "FAIL"
    print(f"  {status}: {rec.get('name', '?'):10s} — {reason}")

In [ ]:
# *args and **kwargs — flexible arguments

# *args: accept any number of positional arguments
def total(*values):
    return sum(values)

print(total(10, 20, 30))      # 60
print(total(5, 15))            # 20

# **kwargs: accept any number of keyword arguments
def build_record(**fields):
    return fields

record = build_record(name="Alice", age=30, city="NYC")
print(record)  # {'name': 'Alice', 'age': 30, 'city': 'NYC'}

In [ ]:
# Putting it together: a reusable cleaning function

def clean_record(raw):
    """Clean a raw record from CSV input."""
    return {
        "name": raw["name"].strip().title(),
        "email": raw["email"].strip().lower(),
        "amount": float(raw["amount"].replace("$", "").replace(",", "")),
        "active": raw["active"].lower() in ("true", "yes", "1"),
    }

# Test
raw = {"name": "  ALICE  ", "email": "Alice@MAIL.COM", "amount": "$1,200.50", "active": "Yes"}
print(clean_record(raw))

**Try it:** Write two functions:

1. `classify_amount(amount)` — returns `"low"` if < 100, `"medium"` if 100-500, `"high"` if > 500
2. `summarize_records(records)` — takes a list of dicts (each with `"amount"` key), returns a dict with `total`, `count`, `average`

In [ ]:
# Your code here

def classify_amount(amount):
    pass

def summarize_records(records):
    pass

# Test classify_amount
print(classify_amount(50))    # low
print(classify_amount(250))   # medium
print(classify_amount(800))   # high

# Test summarize_records
data = [{"amount": 100}, {"amount": 200}, {"amount": 300}]
print(summarize_records(data))  # {'total': 600, 'count': 3, 'average': 200.0}

---

## 4. Lambda Functions

Small anonymous functions. Useful for quick one-liner transformations, especially with `map()`, `filter()`, `sorted()`.

```python
lambda arguments: expression
```

In [ ]:
# Lambda basics

# Regular function
def double(x):
    return x * 2

# Same thing as lambda
double_l = lambda x: x * 2

print(double(5))    # 10
print(double_l(5))  # 10

In [ ]:
# Where lambdas actually shine: with map, filter, sorted

amounts = [120.50, 89.99, 250.00, 45.30, 180.75]

# map — apply a transformation to every item
with_tax = list(map(lambda x: round(x * 1.08, 2), amounts))
print(f"With tax: {with_tax}")

# filter — keep only items that match a condition
big_orders = list(filter(lambda x: x > 100, amounts))
print(f"Over $100: {big_orders}")

# sorted — sort by a custom key
records = [
    {"name": "Alice", "amount": 250},
    {"name": "Bob", "amount": 80},
    {"name": "Charlie", "amount": 180},
]

by_amount = sorted(records, key=lambda r: r["amount"], reverse=True)
for r in by_amount:
    print(f"  {r['name']}: ${r['amount']}")

In [ ]:
# Note: list comprehensions are often cleaner than map/filter

# These are equivalent:
result_map = list(map(lambda x: x * 2, [1, 2, 3]))
result_comp = [x * 2 for x in [1, 2, 3]]
print(result_map)   # [2, 4, 6]
print(result_comp)  # [2, 4, 6]

# Use whichever reads better. For simple transforms, comprehensions win.
# For complex sorting keys, lambda with sorted() is the way to go.

**Try it:**

```python
products = [
    {"name": "Laptop", "price": 999, "stock": 5},
    {"name": "Mouse", "price": 30, "stock": 100},
    {"name": "Monitor", "price": 350, "stock": 0},
    {"name": "Keyboard", "price": 80, "stock": 45},
]
```

1. Use `filter` + lambda to get products where `stock > 0`
2. Use `sorted` + lambda to sort by price (cheapest first)
3. Use a list comprehension to get just the names of in-stock products

In [ ]:
products = [
    {"name": "Laptop", "price": 999, "stock": 5},
    {"name": "Mouse", "price": 30, "stock": 100},
    {"name": "Monitor", "price": 350, "stock": 0},
    {"name": "Keyboard", "price": 80, "stock": 45},
]

# Your code here


---

## Lab Exercises

---

### Lab 1: Filtering Functions

Build reusable functions that filter datasets based on different criteria.

In [ ]:
# Example: a generic filter function

def filter_by_field(records, field, value):
    """Return records where field equals value."""
    return [r for r in records if r.get(field) == value]

def filter_above(records, field, threshold):
    """Return records where field > threshold."""
    return [r for r in records if r.get(field, 0) > threshold]


employees = [
    {"name": "Alice", "dept": "engineering", "salary": 95000},
    {"name": "Bob", "dept": "data", "salary": 82000},
    {"name": "Charlie", "dept": "engineering", "salary": 78000},
    {"name": "Diana", "dept": "data", "salary": 91000},
    {"name": "Eve", "dept": "engineering", "salary": 105000},
]

eng_team = filter_by_field(employees, "dept", "engineering")
print("Engineering:")
for e in eng_team:
    print(f"  {e['name']}: ${e['salary']:,}")

high_earners = filter_above(employees, "salary", 90000)
print("\nSalary > $90k:")
for e in high_earners:
    print(f"  {e['name']}: ${e['salary']:,}")

**Your turn:** Write these filter functions:

1. `filter_by_range(records, field, min_val, max_val)` — returns records where field is between min and max (inclusive)
2. `filter_by_values(records, field, allowed_values)` — returns records where field is one of the allowed values
3. `exclude_nulls(records, field)` — returns records where field is not None and not empty string

Test each function with the employees data.

In [ ]:
employees = [
    {"name": "Alice", "dept": "engineering", "salary": 95000, "rating": 4.5},
    {"name": "Bob", "dept": "data", "salary": 82000, "rating": None},
    {"name": "Charlie", "dept": "engineering", "salary": 78000, "rating": 3.8},
    {"name": "Diana", "dept": "data", "salary": 91000, "rating": 4.2},
    {"name": "Eve", "dept": "hr", "salary": 65000, "rating": ""},
]

# Your code here

def filter_by_range(records, field, min_val, max_val):
    pass

def filter_by_values(records, field, allowed_values):
    pass

def exclude_nulls(records, field):
    pass

# Test: salary between 80k and 95k
# Test: dept in ["engineering", "data"]
# Test: exclude null/empty ratings


---

### Lab 2: Aggregation Functions

Build functions that aggregate data — sum, average, count, group by.

In [ ]:
# Example: group_by function

def group_by(records, key_field):
    """Group records by a field. Returns a dict of lists."""
    groups = {}
    for record in records:
        key = record[key_field]
        if key not in groups:
            groups[key] = []
        groups[key].append(record)
    return groups

def aggregate(records, field):
    """Calculate sum, count, avg, min, max for a numeric field."""
    values = [r[field] for r in records if r.get(field) is not None]
    if not values:
        return {"count": 0}
    return {
        "count": len(values),
        "sum": sum(values),
        "avg": round(sum(values) / len(values), 2),
        "min": min(values),
        "max": max(values),
    }


sales = [
    {"region": "north", "product": "laptop", "amount": 999},
    {"region": "south", "product": "mouse", "amount": 30},
    {"region": "north", "product": "keyboard", "amount": 80},
    {"region": "south", "product": "laptop", "amount": 999},
    {"region": "north", "product": "monitor", "amount": 350},
    {"region": "east", "product": "mouse", "amount": 30},
]

# Group by region and aggregate
by_region = group_by(sales, "region")
for region, records in by_region.items():
    stats = aggregate(records, "amount")
    print(f"{region}: {stats}")

**Your turn:** Write these aggregation functions:

1. `count_by(records, field)` — returns a dict counting occurrences of each value (like a frequency table)
2. `sum_by(records, group_field, sum_field)` — returns a dict with sum of `sum_field` grouped by `group_field`

Test with the sales data above.

In [ ]:
sales = [
    {"region": "north", "product": "laptop", "amount": 999},
    {"region": "south", "product": "mouse", "amount": 30},
    {"region": "north", "product": "keyboard", "amount": 80},
    {"region": "south", "product": "laptop", "amount": 999},
    {"region": "north", "product": "monitor", "amount": 350},
    {"region": "east", "product": "mouse", "amount": 30},
]

def count_by(records, field):
    pass

def sum_by(records, group_field, sum_field):
    pass

# Test count_by: how many sales per region?
# Expected: {'north': 3, 'south': 2, 'east': 1}

# Test count_by: how many sales per product?

# Test sum_by: total amount per region?
# Expected: {'north': 1429, 'south': 1029, 'east': 30}


---

### Lab 3: Mini Project — Sales Data Summarizer

Put it all together. Given a raw sales dataset, build a complete summarizer that:
1. Cleans the data
2. Filters out invalid records
3. Groups and aggregates
4. Prints a nice summary report

In [ ]:
# Raw sales data — needs cleaning

raw_sales = [
    {"date": "2024-01-15", "product": "  Laptop  ", "quantity": "2", "unit_price": "$999.99", "region": "NORTH"},
    {"date": "2024-01-15", "product": "Mouse", "quantity": "10", "unit_price": "$29.99", "region": "south"},
    {"date": "2024-01-16", "product": " KEYBOARD ", "quantity": "5", "unit_price": "$79.50", "region": "North"},
    {"date": "2024-01-16", "product": "Monitor", "quantity": "-1", "unit_price": "$349.99", "region": "east"},
    {"date": "2024-01-17", "product": "Laptop", "quantity": "1", "unit_price": "$999.99", "region": "South"},
    {"date": "2024-01-17", "product": "  headphones ", "quantity": "3", "unit_price": "$59.99", "region": "NORTH"},
    {"date": "2024-01-18", "product": "Mouse", "quantity": "0", "unit_price": "$29.99", "region": "east"},
    {"date": "2024-01-18", "product": "Laptop", "quantity": "1", "unit_price": "$999.99", "region": "north"},
]

**Your tasks:**

Write these functions and use them to produce the final summary.

**Step 1:** `clean_sale(raw)` — clean one record:
- Strip + title case `product`
- Lowercase `region`
- Convert `quantity` to int, `unit_price` to float (remove $)
- Add `total = quantity * unit_price`

**Step 2:** `is_valid(sale)` — return True if quantity > 0

**Step 3:** Process the pipeline:
- Clean all records
- Filter out invalid ones
- Group by region and by product
- Calculate totals per group

**Step 4:** Print a summary report like:
```
=== Sales Summary Report ===
Total records: 8
Valid records: 6
Invalid records: 2

Revenue by Region:
  north: $3,277.44
  south: $1,299.88
  ...

Revenue by Product:
  Laptop: $2,999.97
  ...

Grand Total: $X,XXX.XX
```

In [ ]:
# Step 1: cleaning function
def clean_sale(raw):
    pass

# Step 2: validation function
def is_valid(sale):
    pass

# Step 3: process the pipeline
# - clean all records
# - filter valid ones
# - group by region, group by product

# Step 4: print summary report


---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| if-else | Use for data validation, record routing, status checks |
| for loops | Iterate over batches of records, files, configs |
| while loops | Retries, polling, waiting for conditions |
| List comprehensions | One-liner for transforming/filtering lists |
| Functions | Write once, reuse everywhere — clean, validate, transform |
| Lambda | Quick throwaway functions for sort keys, map/filter |

**Key patterns:**
- Validate records with functions that return `(is_valid, reason)`
- Use `continue` to skip bad records in a loop
- Group-by + aggregate = the most common reporting pattern
- Build small, focused functions — one job per function

**Next session:** File Handling & Logging — reading/writing CSV, JSON, TXT files and adding logging to your pipelines.